In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# ============================================================
#  Configuration
# ============================================================

SEARCH_URL   = "https://lequotidien.sn/?s=accident&paged={page}"
ANNEES       = range(2018, 2026)
MOIS         = range(1, 13)
MAX_PAGES    = 50
SLEEP_MIN    = 2.0
SLEEP_MAX    = 4.0
FICHIER_CSV  = "accidents_lequotidien.csv"
BASE_URL     = "https://lequotidien.sn"

MOTS_CLES = [
    "accident de la route", "accident mortel", "accident de circulation",
    "collision mortelle", "collision entre", "carambolage",
    "percuté", "renversé", "crash", "jakarta",
    "morts dans un accident", "blessés dans un accident",
    "accident tragique", "accident grave", "accident sur",
    "tués dans un accident", "victime d'un accident",
]

MOTS_EXCLUSION = [
    "accident vasculaire", "accident diplomatique", "par accident",
    "can 2025", "can 2026", "football", "new york",
    "allemagne", "iran", "corée", "mali :", "maroc",
    "ukraine", "russie", "chuck norris",
]


# ============================================================
#  Navigateur Selenium
# ============================================================

def init_driver() -> webdriver.Chrome:
    """Lance Chrome headless avec anti-détection."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def get_page(driver, url: str) -> BeautifulSoup | None:
    """Charge une URL avec Selenium, simule comportement humain."""
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 3);")
        time.sleep(random.uniform(0.5, 1.5))
        return BeautifulSoup(driver.page_source, "html.parser")
    except Exception as e:
        print(f"    ⚠️  ({url}) : {e}")
        return None


# ============================================================
#  Extraction
# ============================================================

def extraire_liens(soup: BeautifulSoup) -> list[dict]:
    resultats = []
    cards = soup.find_all("article")
    if cards:
        for card in cards:
            titre_tag = card.find(["h1", "h2", "h3"])
            a_tag = card.find("a", href=True)
            if not titre_tag or not a_tag:
                continue
            titre = titre_tag.text.strip()
            lien = a_tag["href"]
            if not lien.startswith("http"):
                lien = BASE_URL + lien
            if titre and lien:
                resultats.append({"title": titre, "link": lien})
    else:
        for tag in soup.find_all(["h2", "h3"]):
            a = tag.find("a", href=True)
            if not a:
                continue
            titre = a.text.strip()
            lien = a["href"]
            if not lien.startswith("http"):
                lien = BASE_URL + lien
            if titre and len(titre) > 15:
                resultats.append({"title": titre, "link": lien})
    return resultats


def filtrer(articles: list[dict]) -> list[dict]:
    return [
        art for art in articles
        if any(m in art["title"].lower() for m in MOTS_CLES)
        and not any(m in art["title"].lower() for m in MOTS_EXCLUSION)
    ]


def extraire_contenu(driver, url: str) -> dict:
    soup = get_page(driver, url)
    if not soup:
        return {"date": "", "description": "", "localisation": ""}

    date = ""
    date_tag = soup.find("time")
    if date_tag:
        date = date_tag.get("datetime", date_tag.text.strip())
    if not date:
        for prop in ["article:published_time", "og:updated_time"]:
            meta = soup.find("meta", {"property": prop})
            if meta:
                date = meta.get("content", "")
                break

    contenu_div = (
        soup.find("div", class_=lambda c: c and "entry-content" in (c or "")) or
        soup.find("div", class_=lambda c: c and "article-content" in (c or "")) or
        soup.find("article")
    )
    paragraphes = contenu_div.find_all("p") if contenu_div else soup.find_all("p")
    vus, uniques = set(), []
    for p in paragraphes:
        t = p.text.strip()
        if len(t) > 40 and t not in vus:
            vus.add(t)
            uniques.append(t)

    localisation = ""
    h1 = soup.find("h1")
    if h1 and ":" in h1.text:
        localisation = h1.text.split(":")[0].strip()

    return {"date": date, "description": " ".join(uniques), "localisation": localisation}


# ============================================================
#  Programme principal
# ============================================================

def main():
    print("=" * 60)
    print("  Scraper Le Quotidien — Selenium (anti-CleanTalk)")
    print("=" * 60)

    driver = init_driver()
    tous_liens = []

    try:
        # Visite page d'accueil pour cookies
        print("\n🍪 Initialisation des cookies...")
        get_page(driver, BASE_URL)
        print("   ✅ Prêt")

        # --- Stratégie 1 : Recherche ---
        print(f"\n🔗 Recherche par mot-clé (max {MAX_PAGES} pages)...")
        for page in range(1, MAX_PAGES + 1):
            url = SEARCH_URL.replace("{page}", str(page))
            soup = get_page(driver, url)
            if not soup:
                break
            liens = extraire_liens(soup)
            if not liens:
                print(f"   Page {page} : vide → arrêt")
                break
            filtres = filtrer(liens)
            tous_liens.extend(filtres)
            print(f"   Page {page} : {len(liens)} articles, {len(filtres)} retenus")

        # --- Stratégie 2 : Archives mensuelles ---
        print(f"\n📅 Archives mensuelles (2018 → 2025)...")
        for annee in ANNEES:
            for mois in MOIS:
                url = f"{BASE_URL}/{annee}/{mois:02d}/"
                soup = get_page(driver, url)
                if not soup:
                    continue
                liens = extraire_liens(soup)
                filtres = filtrer(liens)
                if filtres:
                    tous_liens.extend(filtres)
                    print(f"   {annee}/{mois:02d} : {len(filtres)} retenus / {len(liens)}")

    finally:
        driver.quit()
        print("\n🔒 Navigateur fermé")

    # Dédoublonner
    seen, liens_uniques = set(), []
    for art in tous_liens:
        if art["link"] not in seen:
            seen.add(art["link"])
            liens_uniques.append(art)

    print(f"\n✅ Total liens uniques : {len(liens_uniques)}")
    if not liens_uniques:
        print("❌ Aucun article trouvé.")
        return

    # Scraper chaque article
    print(f"\n📰 Extraction du contenu...\n")
    driver = init_driver()
    resultats = []
    try:
        get_page(driver, BASE_URL)
        for i, art in enumerate(liens_uniques, 1):
            print(f"  [{i}/{len(liens_uniques)}] {art['title'][:65]}...")
            contenu = extraire_contenu(driver, art["link"])
            resultats.append({
                "date":         contenu["date"],
                "localisation": contenu["localisation"],
                "titre":        art["title"],
                "description":  contenu["description"],
                "lien":         art["link"],
                "source":       "Le Quotidien",
            })
    finally:
        driver.quit()

    df = pd.DataFrame(resultats)
    if df.empty:
        print("\n⚠️  Aucun article extrait.")
        return

    df["description"] = df["description"].str.replace(r"\s+", " ", regex=True).str.strip()
    df.to_csv(FICHIER_CSV, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 60)
    print(f"  ✅  Fichier créé      : {FICHIER_CSV}")
    print(f"  📰  Articles extraits : {len(df)}")
    print("=" * 60)
    print(df[["date", "localisation", "titre"]].head(10).to_string(index=False))


if __name__ == "__main__":
    main()

Recuperation avec Methode 2

In [ ]:
import os

# Liste étendue pour le Sénégal
VILLES_SENEGAL = [
    "Dakar", "Thiès", "Diourbel", "Kaolack", "Saint-Louis", "Ziguinchor", 
    "Tambacounda", "Kaffrine", "Louga", "Fatick", "Matam", "Kolda", "Sédhiou", "Kédougou",
    "Mbour", "Touba", "Rufisque", "Pikine", "Guédiawaye", "Bignona", "Richard-Toll", "Bambey"
]

EXCLUSIONS_PAYS = ["mali", "france", "ukraine", "russie", "maroc", "espagne", "côte d'ivoire", "guinée"]

def est_un_accident_senegalais(titre):
    titre_l = titre.lower()
    # 1. Vérifier les exclusions directes
    if any(p in titre_l for p in EXCLUSIONS_PAYS):
        return False
    # 2. Vérifier les mots-clés routiers
    if not any(m in titre_l for m in MOTS_CLES):
        return False
    return True

def sauvegarde_ligne(data, fichier):
    """Sauvegarde l'article immédiatement dans le CSV."""
    df_temp = pd.DataFrame([data])
    file_exists = os.path.isfile(fichier)
    df_temp.to_csv(fichier, mode='a', index=False, header=not file_exists, encoding="utf-8-sig")

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

# ============================================================
# 1. CONFIGURATION & FILTRAGE (À placer en premier)
# ============================================================

BASE_URL     = "https://lequotidien.sn"
# On cible la catégorie Société qui contient les faits divers
SEARCH_URL   = "https://lequotidien.sn/category/societe/page/{page}/"
FICHIER_CSV  = "accidents_lequotidien.csv"

MAX_PAGES    = 50
SLEEP_MIN    = 2.0
SLEEP_MAX    = 5.0

# Mots-clés pour identifier un accident de la route
MOTS_CLES = [
    "accident", "collision", "choc", "renversé", "percuté", "tué", 
    "mortel", "mort", "drame", "fauché", "blessé", "jakarta", "carambolage"
]

# Exclusions pour éviter le "bruit" (International et hors-sujet)
EXCLUSIONS_PAYS = [
    "mali", "france", "ukraine", "russie", "maroc", "espagne", 
    "italie", "usa", "états-unis", "can 20", "football", "diplomatique"
]

# Liste des villes pour renforcer la détection locale
VILLES_SENEGAL = [
    "Dakar", "Thiès", "Diourbel", "Kaolack", "Saint-Louis", "Ziguinchor", 
    "Tambacounda", "Kaffrine", "Louga", "Fatick", "Matam", "Kolda", "Sédhiou", "Kédougou",
    "Mbour", "Touba", "Rufisque", "Pikine", "Guédiawaye", "Bignona", "Bambey"
]

# ============================================================
# 2. FONCTIONS DE FILTRAGE ET SAUVEGARDE
# ============================================================

def est_un_accident_senegalais(titre):
    """Vérifie si le titre correspond à un accident au Sénégal."""
    if not titre: return False
    titre_l = titre.lower()
    
    # 1. Exclusion si pays étranger ou sport
    if any(p in titre_l for p in EXCLUSIONS_PAYS):
        return False
    
    # 2. Inclusion si mot-clé d'accident présent
    if any(m in titre_l for m in MOTS_CLES):
        return True
        
    return False

def sauvegarde_ligne(data, fichier):
    """Sauvegarde l'article immédiatement pour éviter les pertes de données."""
    df_temp = pd.DataFrame([data])
    file_exists = os.path.isfile(fichier)
    df_temp.to_csv(fichier, mode='a', index=False, header=not file_exists, encoding="utf-8-sig")

# ============================================================
# 3. MOTEUR SELENIUM & EXTRACTION
# ============================================================

def init_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(f"user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    driver = webdriver.Chrome(options=options)
    return driver

def get_page(driver, url):
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))
        # Scroll pour déclencher le chargement des images/contenus
        driver.execute_script("window.scrollTo(0, 1000);")
        return BeautifulSoup(driver.page_source, "html.parser")
    except Exception as e:
        print(f"  ⚠️ Erreur sur {url}: {e}")
        return None

def extraire_liens(soup):
    """Extrait les titres et liens avec un sélecteur plus large."""
    resultats = []
    # On cherche dans les balises de titres d'articles classiques de WordPress
    articles = soup.find_all(["h2", "h3"], class_=lambda x: x is not None)
    if not articles:
        articles = soup.find_all(["h2", "h3"]) # Repli si pas de classe

    for tag in articles:
        a_tag = tag.find("a", href=True)
        if a_tag:
            titre = a_tag.text.strip()
            lien = a_tag["href"]
            if len(titre) > 15:
                resultats.append({"title": titre, "link": lien})
    return resultats

def extraire_contenu(driver, url):
    """Récupère la date et le texte complet de l'article."""
    soup = get_page(driver, url)
    if not soup: return {"date": "", "description": "", "localisation": ""}

    # Date
    date_tag = soup.find("time")
    date = date_tag.get("datetime", date_tag.text.strip()) if date_tag else ""

    # Corps de l'article
    div_content = soup.find("div", class_="entry-content") or soup.find("article")
    paragraphes = div_content.find_all("p") if div_content else []
    texte = " ".join([p.text.strip() for p in paragraphes if len(p.text) > 30])

    # Tentative de localisation (souvent au début du texte en gras)
    localisation = ""
    if paragraphes and paragraphes[0].find("strong"):
        localisation = paragraphes[0].find("strong").text.replace(":", "").strip()

    return {"date": date, "description": texte, "localisation": localisation}

# ============================================================
# 4. BOUCLE PRINCIPALE (RÉSILIENTE)
# ============================================================

def main():
    print("="*60)
    print("🚀 Démarrage du Scraper Le Quotidien (Spécial Sénégal)")
    print("="*60)

    driver = init_driver()
    liens_valides = []

    try:
        # PHASE 1 : COLLECTE DES LIENS
        for page in range(1, MAX_PAGES + 1):
            url = SEARCH_URL.replace("{page}", str(page))
            print(f"🔎 Analyse Page {page}...", end="\r")
            soup = get_page(driver, url)
            if not soup: break
            
            liens_page = extraire_liens(soup)
            if not liens_page: 
                print(f"\nFin des résultats à la page {page}")
                break

            for art in liens_page:
                if est_un_accident_senegalais(art['title']):
                    liens_valides.append(art)
        
        print(f"\n✅ Collecte terminée : {len(liens_valides)} articles potentiels trouvés.")

        # PHASE 2 : CHECKPOINT (Éviter les doublons)
        liens_deja_faits = set()
        if os.path.exists(FICHIER_CSV):
            df_existant = pd.read_csv(FICHIER_CSV)
            liens_deja_faits = set(df_existant['lien'].tolist())
        
        liens_a_traiter = [l for l in liens_valides if l['link'] not in liens_deja_faits]
        print(f"📥 {len(liens_a_traiter)} nouveaux articles à extraire.")

    finally:
        driver.quit()

    # PHASE 3 : EXTRACTION ET SAUVEGARDE UNITAIRE
    if liens_a_traiter:
        driver = init_driver()
        try:
            for i, art in enumerate(liens_a_traiter, 1):
                print(f"📝 [{i}/{len(liens_a_traiter)}] {art['title'][:60]}...")
                info = extraire_contenu(driver, art['link'])
                
                donnees_finales = {
                    "date": info["date"],
                    "localisation": info["localisation"],
                    "titre": art["title"],
                    "description": info["description"],
                    "lien": art["link"],
                    "source": "Le Quotidien"
                }
                
                sauvegarde_ligne(donnees_finales, FICHIER_CSV)
                time.sleep(random.uniform(1, 3))
        finally:
            driver.quit()

    print("\n✨ Opération terminée. Données sauvegardées dans :", FICHIER_CSV)

if __name__ == "__main__":
    main()

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import unicodedata
import re

# ============================================================
# 1. CONFIGURATION & FILTRAGE
# ============================================================

BASE_URL    = "https://lequotidien.sn"
SEARCH_URL  = "https://lequotidien.sn/category/societe/page/{page}/"
FICHIER_CSV = "accidents_lequotidien1.csv"

MAX_PAGES  = 50
SLEEP_MIN  = 2.0
SLEEP_MAX  = 5.0

MOTS_CLES = [
    "accident", "collision", "choc", "renversé", "percuté",
    "tué", "mortel", "fauché", "blessé", "jakarta",
    "carambolage", "dérape"
]

EXCLUSIONS = [
    "mali", "france", "ukraine", "russie", "maroc", "espagne",
    "usa", "can 20", "football", "noyade", "pirogue",
    "meurtre", "agression"
]

CONTEXTE_ROUTIER = [
    "véhicule", "voiture", "moto", "car", "bus", "camion",
    "route", "rn1", "rn2", "autoroute", "chaussée", "piste",
    "vitesse", "pneu", "volant"
]

# ============================================================
# 2. DICTIONNAIRE DE LIEUX DU SÉNÉGAL (étendu)
# ============================================================

LIEUX_SENEGAL = {
    # --- Régions ---
    "Dakar": ["dakar"],
    "Thiès": ["thiès", "thies"],
    "Diourbel": ["diourbel"],
    "Kaolack": ["kaolack"],
    "Saint-Louis": ["saint-louis", "saint louis"],
    "Ziguinchor": ["ziguinchor"],
    "Tambacounda": ["tambacounda"],
    "Kaffrine": ["kaffrine"],
    "Louga": ["louga"],
    "Fatick": ["fatick"],
    "Matam": ["matam"],
    "Kolda": ["kolda"],
    "Sédhiou": ["sédhiou", "sedhiou"],
    "Kédougou": ["kédougou", "kedougou"],

    # --- Villes et communes ---
    "Mbour": ["mbour"],
    "Touba": ["touba"],
    "Rufisque": ["rufisque"],
    "Pikine": ["pikine"],
    "Guédiawaye": ["guédiawaye", "guediawaye"],
    "Bignona": ["bignona"],
    "Bambey": ["bambey"],
    "Vélingara": ["vélingara", "velingara"],
    "Tivaouane": ["tivaouane"],
    "Joal": ["joal"],
    "Saly": ["saly"],
    "Bargny": ["bargny"],
    "Diamniadio": ["diamniadio"],
    "Mbacké": ["mbacké", "mbacke"],
    "Dara": ["dara"],
    "Linguère": ["linguère", "linguere"],
    "Podor": ["podor"],
    "Richard-Toll": ["richard-toll", "richard toll"],
    "Dagana": ["dagana"],
    "Bakel": ["bakel"],
    "Kanel": ["kanel"],
    "Ranérou": ["ranérou", "ranerou"],
    "Gossas": ["gossas"],
    "Sokone": ["sokone"],
    "Foundiougne": ["foundiougne"],
    "Ndoffane": ["ndoffane"],
    "Nioro": ["nioro"],
    "Birkelane": ["birkelane"],
    "Malème-Hodar": ["malème-hodar", "maleme hodar"],
    "Oussouye": ["oussouye"],
    "Goudomp": ["goudomp"],
    "Bounkiling": ["bounkiling"],
    "Médina Yoro Foulah": ["médina yoro foulah", "medina yoro foulah"],
    "Saraya": ["saraya"],
    "Salémata": ["salémata", "salemata"],
    "Yeumbeul": ["yeumbeul"],
    "Thiaroye": ["thiaroye"],
    "Parcelles Assainies": ["parcelles assainies", "parcelles"],
    "Médina": ["médina", "medina"],
    "Ouakam": ["ouakam"],
    "Ngor": ["ngor"],
    "Yoff": ["yoff"],
    "Cambérène": ["cambérène", "camberene"],
    "Hann": ["hann"],
    "Grand Yoff": ["grand yoff"],
    "Almadies": ["almadies"],
    "Sicap": ["sicap"],
    "Mermoz": ["mermoz"],
    "Sacré-Coeur": ["sacré-coeur", "sacre coeur"],
    "Dieuppeul": ["dieuppeul"],

    # --- Axes routiers ---
    "RN1": ["rn1", "route nationale 1", "nationale 1"],
    "RN2": ["rn2", "route nationale 2", "nationale 2"],
    "RN6": ["rn6", "route nationale 6"],
    "Autoroute à péage": ["autoroute à péage", "autoroute a peage", "vdn"],
    "Route de Thiès": ["route de thiès", "route de thies"],
    "Route de Mbour": ["route de mbour"],
    "Route de Touba": ["route de touba"],
    "Route de Kaolack": ["route de kaolack"],
    "Corniche": ["corniche"],
    "Diamniadio": ["diamniadio"],
}

# ============================================================
# 3. FONCTIONS DE LOGIQUE
# ============================================================

def normaliser(texte):
    """Supprime les accents et met en minuscules pour une comparaison robuste."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', texte.lower())
        if unicodedata.category(c) != 'Mn'
    )

def trouver_ville_dans_texte(titre, description):
    """
    Cherche d'abord dans le titre (priorité haute),
    puis dans titre + description ensemble.
    Utilise la normalisation pour gérer les accents.
    """
    titre_normalise      = normaliser(str(titre))
    texte_complet_norme  = normaliser(f"{str(titre)} {str(description)}")

    # Priorité 1 : titre seul
    for nom_ville, variantes in LIEUX_SENEGAL.items():
        for variante in variantes:
            v_norm = normaliser(variante)
            pattern = r'\b' + re.escape(v_norm) + r'\b'
            if re.search(pattern, titre_normalise):
                return nom_ville

    # Priorité 2 : titre + description
    for nom_ville, variantes in LIEUX_SENEGAL.items():
        for variante in variantes:
            v_norm = normaliser(variante)
            pattern = r'\b' + re.escape(v_norm) + r'\b'
            if re.search(pattern, texte_complet_norme):
                return nom_ville

    return "Sénégal (Général)"

def est_un_accident_routier_senegal(titre):
    """Filtre strictement pour le contexte routier au Sénégal."""
    t_low = titre.lower()
    if any(p in t_low for p in EXCLUSIONS):
        return False
    a_mot_cle  = any(m in t_low for m in MOTS_CLES)
    a_contexte = any(c in t_low for c in CONTEXTE_ROUTIER)
    return a_mot_cle or (a_mot_cle and a_contexte)

def sauvegarde_ligne(data, fichier):
    """Sauvegarde une ligne dans le CSV de façon incrémentale."""
    df_temp    = pd.DataFrame([data])
    file_exists = os.path.isfile(fichier)
    df_temp.to_csv(fichier, mode='a', index=False,
                   header=not file_exists, encoding="utf-8-sig")

# ============================================================
# 4. NAVIGATION ET EXTRACTION
# ============================================================

def init_driver():
    """Initialise Chrome avec des options de stabilité."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(30)
    return driver

def get_page(driver, url):
    """Charge une page et retourne le BeautifulSoup, ou None en cas d'erreur."""
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))
        return BeautifulSoup(driver.page_source, "html.parser")
    except Exception as e:
        print(f"⚠️  Impossible de charger {url} : {e}")
        return None

def extraire_liens(soup):
    """Extrait les titres et liens d'articles depuis une page de liste."""
    resultats = []
    for tag in soup.find_all(["h2", "h3"]):
        a = tag.find("a", href=True)
        if a and "/category/" not in a['href'] and len(a.text.strip()) > 20:
            resultats.append({
                "title": a.text.strip(),
                "link":  a['href']
            })
    return resultats

def extraire_contenu(driver, url):
    """
    Ouvre un article et extrait la date + le texte complet.
    Retourne un dict ou None en cas d'échec.
    """
    soup = get_page(driver, url)
    if not soup:
        return None

    # --- Date ---
    dt_tag = soup.find("time")
    date   = dt_tag.get("datetime", dt_tag.text.strip()) if dt_tag else ""

    # --- Texte de l'article ---
    content = soup.find("div", class_="entry-content") or soup.find("article")
    paras   = content.find_all("p") if content else []
    texte   = " ".join([p.text.strip() for p in paras if len(p.text) > 30])

    return {"date": date, "description": texte}

# ============================================================
# 5. PROGRAMME PRINCIPAL
# ============================================================

def main():
    print("🚀 Initialisation du scraper sécurisé...")
    driver          = init_driver()
    liens_potentiels = []

    try:
        # ----------------------------------------------------------
        # PHASE 1 : COLLECTE DES LIENS
        # ----------------------------------------------------------
        print(f"\n📋 Phase 1 : Collecte des liens sur {MAX_PAGES} pages...")
        for p in range(1, MAX_PAGES + 1):
            print(f"   🔎 Analyse page {p}/{MAX_PAGES}...", end="\r")
            soup = get_page(driver, SEARCH_URL.format(page=p))
            if not soup:
                print(f"\n⚠️  Page {p} inaccessible, arrêt de la collecte.")
                break

            liens = extraire_liens(soup)
            for l in liens:
                if est_un_accident_routier_senegal(l['title']):
                    liens_potentiels.append(l)

        print(f"\n✅ {len(liens_potentiels)} articles filtrés trouvés.")

        # ----------------------------------------------------------
        # PHASE 2 : DÉDOUBLONNAGE
        # ----------------------------------------------------------
        print("\n🔁 Phase 2 : Vérification des doublons...")
        deja_scrappes = set()
        if os.path.exists(FICHIER_CSV):
            try:
                deja_scrappes = set(pd.read_csv(FICHIER_CSV)['lien'].tolist())
                print(f"   📂 {len(deja_scrappes)} articles déjà présents dans le CSV.")
            except Exception:
                print("   ⚠️  CSV existant illisible, on repart de zéro.")

        a_traiter = [l for l in liens_potentiels if l['link'] not in deja_scrappes]
        print(f"   📥 {len(a_traiter)} nouveaux articles à extraire.")

    finally:
        driver.quit()

    # ----------------------------------------------------------
    # PHASE 3 : EXTRACTION ROBUSTE ARTICLE PAR ARTICLE
    # ----------------------------------------------------------
    if not a_traiter:
        print("\n✨ Aucun nouvel article à traiter. Tout est à jour !")
        return

    print(f"\n📝 Phase 3 : Extraction du contenu des {len(a_traiter)} articles...")
    driver  = init_driver()
    succes  = 0
    echecs  = 0

    for i, art in enumerate(a_traiter, 1):
        try:
            print(f"   [{i:>3}/{len(a_traiter)}] {art['title'][:60]}...")
            info = extraire_contenu(driver, art['link'])

            # Gestion du crash driver
            if info is None:
                print("   🔄 Crash navigateur détecté. Relance en cours...")
                driver.quit()
                time.sleep(5)
                driver = init_driver()
                info   = extraire_contenu(driver, art['link'])
                if not info:
                    echecs += 1
                    print(f"   ❌ Article {i} ignoré après relance.")
                    continue

            # Localisation améliorée
            ville = trouver_ville_dans_texte(art['title'], info['description'])

            sauvegarde_ligne({
                "date":        info["date"],
                "localisation": ville,
                "titre":       art["title"],
                "description": info["description"],
                "lien":        art["link"],
                "source":      "Le Quotidien"
            }, FICHIER_CSV)

            succes += 1
            time.sleep(random.uniform(2, 4))

        except Exception as e:
            echecs += 1
            print(f"   ❌ Erreur article {i} : {e}. On continue...")
            continue

    driver.quit()

    # ----------------------------------------------------------
    # RÉSUMÉ FINAL
    # ----------------------------------------------------------
    print(f"""
╔══════════════════════════════════════╗
║          ✅ SCRAPING TERMINÉ         ║
╠══════════════════════════════════════╣
║  Articles extraits  : {succes:<15} ║
║  Erreurs            : {echecs:<15} ║
║  Fichier CSV        : {FICHIER_CSV:<15} ║
╚══════════════════════════════════════╝
""")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time, random, re, os

# --- CONFIGURATION ---
# L'URL contient maintenant un paramètre {page} pour la boucle
SEARCH_URL_TEMPLATE = "https://lequotidien.sn/page/{page}/?s=accident" 
FICHIER_CSV = "accidents_lequotidien_complet.csv"
MAX_PAGES = 149 

def init_driver():
    options = Options()
    options.add_argument("--start-maximized")
    # Si vous voulez que ça aille plus vite sans voir la fenêtre, 
    # vous pourrez rajouter options.add_argument("--headless") plus tard
    driver = webdriver.Chrome(options=options)
    return driver

# Initialisation du CSV avec entêtes si le fichier n'existe pas
if not os.path.exists(FICHIER_CSV):
    pd.DataFrame(columns=["date", "titre", "lien", "nb_morts", "nb_blesses", "extrait"]).to_csv(FICHIER_CSV, index=False, encoding='utf-8-sig')

driver = init_driver()

try:
    for num_page in range(1, MAX_PAGES + 1):
        print(f"\n--- 📄 SCRAPING PAGE DE RECHERCHE {num_page}/{MAX_PAGES} ---")
        driver.get(SEARCH_URL_TEMPLATE.format(page=num_page))
        time.sleep(random.uniform(4, 6)) # Pause pour laisser la page charger

        # 1. On récupère tous les liens d'articles sur la page de résultats actuelle
        tous_les_liens = driver.find_elements(By.XPATH, "//a[contains(@href, 'accident')]")
        
        articles_de_la_page = []
        for l in tous_les_liens:
            url = l.get_attribute("href")
            titre = l.text.strip()
            # On vérifie que c'est un vrai lien d'article (longueur titre > 15)
            if url and len(titre) > 15 and url not in [x[1] for x in articles_de_la_page]:
                articles_de_la_page.append((titre, url))

        print(f"🔎 {len(articles_de_la_page)} articles identifiés sur cette page.")

        # 2. On visite chaque article trouvé
        for titre, lien in articles_de_la_page:
            print(f"  📖 Analyse de : {titre[:50]}...")
            try:
                driver.get(lien)
                time.sleep(random.uniform(3, 5))
                
                # Extraction du corps de l'article
                try:
                    corps = driver.find_element(By.CLASS_NAME, "entry-content").text
                except:
                    try:
                        corps = driver.find_element(By.TAG_NAME, "article").text
                    except:
                        corps = ""

                if corps:
                    # Extraction des chiffres (Regex améliorée)
                    morts = re.search(r'(\d+)\s+(mort|décès|tué|perdu la vie)', corps.lower())
                    blesses = re.search(r'(\d+)\s+(blessé|hospitalisé|évacué|blessure)', corps.lower())
                    
                    m = morts.group(1) if morts else "0"
                    b = blesses.group(1) if blesses else "0"

                    # Sauvegarde immédiate
                    new_row = pd.DataFrame([{
                        "date": "À nettoyer", 
                        "titre": titre, 
                        "lien": lien, 
                        "nb_morts": m, 
                        "nb_blesses": b, 
                        "extrait": corps[:400].replace('\n', ' ') # On nettoie les sauts de ligne
                    }])
                    new_row.to_csv(FICHIER_CSV, mode='a', index=False, header=False, encoding='utf-8-sig')
                    print(f"    ✅ Sauvegardé : {m} morts / {b} blessés")
            
            except Exception as e:
                print(f"    ❌ Erreur lors de la lecture de l'article : {e}")
                continue

finally:
    driver.quit()
    print(f"\n🏁 TOUTES LES PAGES ONT ÉTÉ TRAITÉES.")
    print(f"Vérifiez votre fichier : {os.path.abspath(FICHIER_CSV)}")

In [ ]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time, random, re, os

SEARCH_URL_TEMPLATE = "https://lequotidien.sn/page/{page}/?s=accident"
FICHIER_CSV = "accidents_lequotidien_1.csv"
MAX_PAGES = 149

def init_driver():
    options = Options()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)
    return driver

def extraire_date(driver):
    """
    Essaie plusieurs sélecteurs pour trouver la date de publication.
    Retourne une chaîne date ou None si introuvable.
    """
    # Stratégie 1 : balise <time> avec attribut datetime (WordPress standard)
    try:
        el = driver.find_element(By.TAG_NAME, "time")
        dt = el.get_attribute("datetime")
        if dt:
            return dt[:10]  # format YYYY-MM-DD
    except:
        pass

    # Stratégie 2 : balise <time> texte visible
    try:
        el = driver.find_element(By.TAG_NAME, "time")
        txt = el.text.strip()
        if txt:
            return txt
    except:
        pass

    # Stratégie 3 : classes CSS typiques WordPress
    selectors = [
        ".entry-date",
        ".post-date",
        ".published",
        ".date",
        ".entry-meta",
        ".post-meta",
        "span.date",
        "span.posted-on",
        ".meta-date",
        ".article-date",
    ]
    for sel in selectors:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except:
            continue

    # Stratégie 4 : chercher dans les meta tags (og:article:published_time)
    try:
        el = driver.find_element(
            By.XPATH,
            "//meta[@property='article:published_time']"
        )
        content = el.get_attribute("content")
        if content:
            return content[:10]
    except:
        pass

    # Stratégie 5 : chercher dans l'URL (ex: /2022/04/15/titre-article/)
    url = driver.current_url
    match = re.search(r'/(\d{4})/(\d{2})/(\d{2})/', url)
    if match:
        return f"{match.group(1)}-{match.group(2)}-{match.group(3)}"

    return "date_non_trouvée"


if not os.path.exists(FICHIER_CSV):
    pd.DataFrame(columns=["date", "titre", "lien", "nb_morts", "nb_blesses", "extrait"]).to_csv(
        FICHIER_CSV, index=False, encoding='utf-8-sig'
    )

driver = init_driver()

try:
    for num_page in range(1, MAX_PAGES + 1):
        print(f"\n--- 📄 PAGE {num_page}/{MAX_PAGES} ---")
        driver.get(SEARCH_URL_TEMPLATE.format(page=num_page))
        time.sleep(random.uniform(4, 6))

        tous_les_liens = driver.find_elements(By.XPATH, "//a[contains(@href, 'accident')]")

        articles_de_la_page = []
        for l in tous_les_liens:
            url = l.get_attribute("href")
            titre = l.text.strip()
            if url and len(titre) > 15 and url not in [x[1] for x in articles_de_la_page]:
                articles_de_la_page.append((titre, url))

        print(f"🔎 {len(articles_de_la_page)} articles identifiés.")

        for titre, lien in articles_de_la_page:
            print(f"  📖 {titre[:60]}...")
            try:
                driver.get(lien)
                time.sleep(random.uniform(3, 5))

                # ── Extraction de la date ──────────────────────────────────
                date = extraire_date(driver)

                # ── Extraction du corps ────────────────────────────────────
                corps = ""
                for selector in ["entry-content", "post-content", "article-content", "content"]:
                    try:
                        corps = driver.find_element(By.CLASS_NAME, selector).text
                        break
                    except:
                        continue
                if not corps:
                    try:
                        corps = driver.find_element(By.TAG_NAME, "article").text
                    except:
                        corps = ""

                if corps:
                    morts   = re.search(r'(\d+)\s+(mort|décès|tué|perdu la vie)', corps.lower())
                    blesses = re.search(r'(\d+)\s+(blessé|hospitalisé|évacué|blessure)', corps.lower())
                    m = morts.group(1)   if morts   else "0"
                    b = blesses.group(1) if blesses else "0"

                    new_row = pd.DataFrame([{
                        "date":      date,
                        "titre":     titre,
                        "lien":      lien,
                        "nb_morts":  m,
                        "nb_blesses": b,
                        "extrait":   corps[:400].replace('\n', ' ')
                    }])
                    new_row.to_csv(FICHIER_CSV, mode='a', index=False, header=False, encoding='utf-8-sig')
                    print(f"    ✅ {date} | {m} morts / {b} blessés")

            except Exception as e:
                print(f"    ❌ Erreur : {e}")
                continue

finally:
    driver.quit()
    print(f"\n🏁 Terminé. Fichier : {os.path.abspath(FICHIER_CSV)}")

In [ ]:
"""
SCRAPER INTELLIGENT — Le Quotidien (Recherche "accident")
Extraction : date, titre, lien, localisation,
             deces, blesses, type_vehicule, description, source
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import unicodedata

# ============================================================
# CONFIGURATION
# ============================================================

BASE_SEARCH_URL = "https://lequotidien.sn/page/{page}/?s=accident"
MAX_PAGES = 150
FICHIER = "accidents_lequotidien_propre.csv"

HEADERS = {"User-Agent": "Mozilla/5.0"}

# ============================================================
# OUTILS TEXTE
# ============================================================

def normaliser(texte):
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(texte).lower())
        if unicodedata.category(c) != 'Mn'
    )

# ============================================================
# LIEUX SIMPLIFIÉS (tu peux enrichir après)
# ============================================================

LIEUX = [
    "dakar","thies","diourbel","kaolack","saint-louis",
    "ziguinchor","tambacounda","kolda","matam",
    "mbour","touba","rufisque","velingara",
    "tivaouane","pikine","guediawaye","bambey"
]

def extraire_lieu(texte):
    texte_n = normaliser(texte)
    for lieu in LIEUX:
        if lieu in texte_n:
            return lieu.title()
    return "Non précisé"

# ============================================================
# EXTRACTION NOMBRES
# ============================================================

def extraire_nombre(texte, mots):
    texte_n = normaliser(texte)
    for mot in mots:
        pattern = rf'(\d+)\s+{mot}'
        match = re.search(pattern, texte_n)
        if match:
            return int(match.group(1))
    return 0

def extraire_deces(texte):
    return extraire_nombre(texte, ["mort","morts","deces","tué","tues"])

def extraire_blesses(texte):
    return extraire_nombre(texte, ["blesse","blesses","blessés","blessées"])

# ============================================================
# VEHICULES
# ============================================================

VEHICULES = {
    "moto": ["moto","jakarta"],
    "bus": ["bus","car rapide","minibus"],
    "camion": ["camion","poids lourd"],
    "voiture": ["voiture","4x4","vehicule"],
    "taxi": ["taxi","sept places"],
}

def extraire_vehicule(texte):
    texte_n = normaliser(texte)
    trouves = []
    for type_v, mots in VEHICULES.items():
        if any(m in texte_n for m in mots):
            trouves.append(type_v)
    return ", ".join(trouves) if trouves else "Non précisé"

# ============================================================
# REQUÊTES
# ============================================================

def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    if r.status_code != 200:
        return None
    return BeautifulSoup(r.text, "html.parser")

def extraire_liens_recherche(soup):
    liens = []
    articles = soup.find_all("h2", class_="entry-title")
    for art in articles:
        a = art.find("a")
        if a:
            liens.append({
                "titre": a.text.strip(),
                "lien": a["href"]
            })
    return liens

def extraire_article(url):
    soup = get_soup(url)
    if not soup:
        return None

    titre = soup.find("h1").text.strip() if soup.find("h1") else ""
    date_tag = soup.find("time")
    date = date_tag.text.strip() if date_tag else ""

    content = soup.find("div", class_="entry-content")
    paragraphs = content.find_all("p") if content else []
    texte = " ".join(p.text.strip() for p in paragraphs)

    return {
        "titre": titre,
        "date": date,
        "description": texte
    }

# ============================================================
# FILTRE ROUTIER
# ============================================================

def est_routier(texte):
    mots_route = [
        "route","rn1","rn2","rn6","autoroute",
        "voiture","moto","bus","camion",
        "jakarta","car rapide","taxi"
    ]
    t = normaliser(texte)
    return any(m in t for m in mots_route)

# ============================================================
# SCRAPING PRINCIPAL
# ============================================================

data = []
liens_vus = set()

for page in range(1, MAX_PAGES + 1):
    print(f"\nPage {page}")
    url = BASE_SEARCH_URL.format(page=page)
    soup = get_soup(url)

    if not soup:
        break

    liens = extraire_liens_recherche(soup)
    if not liens:
        print("Fin des pages.")
        break

    for l in liens:
        if l["lien"] in liens_vus:
            continue
        liens_vus.add(l["lien"])

        article = extraire_article(l["lien"])
        if not article:
            continue

        texte_complet = article["titre"] + " " + article["description"]

        if not est_routier(texte_complet):
            continue

        lieu = extraire_lieu(texte_complet)
        deces = extraire_deces(texte_complet)
        blesses = extraire_blesses(texte_complet)
        vehicule = extraire_vehicule(texte_complet)

        data.append({
            "date": article["date"],
            "titre": article["titre"],
            "lien": l["lien"],
            "localisation": lieu,
            "deces": deces,
            "blesses": blesses,
            "type_vehicule": vehicule,
            "description": article["description"][:1200],
            "source": "Le Quotidien"
        })

        print(f"  ✅ {article['titre'][:60]}")

        time.sleep(1)

df = pd.DataFrame(data)
df.to_csv(FICHIER, index=False, encoding="utf-8-sig")

print("\n===================================")
print("SCRAPING TERMINÉ")
print("Articles récupérés :", len(df))
print("Fichier :", FICHIER)
print("===================================")

In [ ]:
import pandas as pd
df = pd.read_csv("accidents_lequotidien_final.csv")

In [ ]:
print(df.columns)
print(df.head())
print(len(df))

In [ ]:
def extract_city(text):
    if not isinstance(text, str):
        return "Inconnue"
    cities = ["Dakar", "Thiès", "Diourbel", "Kaolack", "Saint-Louis", "Ziguinchor", "Tambacounda", "Kédougou", "Kolda", "Matam", "Louga", "Fatick", "Sédhiou", "Casamance", "Bignona"]

    for city in cities:
        if city.lower() in text.lower():
            return city
    return "Inconnue"

# Extraire d'abord de la description
df["ville"] = df["description"].apply(extract_city)

# Pour les lignes où la ville est "Inconnue", essayer le titre
mask = df["ville"] == "Inconnue"
df.loc[mask, "ville"] = df.loc[mask, "titre"].apply(extract_city)

In [ ]:
df.groupby("ville").size().sort_values(ascending=False)

In [ ]:
df['titre']

In [ ]:
# Dictionnaire complet mis à jour avec les nouveaux indices implicites
mapping_contextuel = {
    "Dakar": [
        "TER", "Vdn", "Corniche", "Pikine", "Guédiawaye", "Rufisque", "Diamniadio", 
        "Colobane", "Fann", "Almadies", "Grand Yoff", "Parcelles Assainies", 
        "Patte d'Oie", "Sébikotane", "Bargny", "Keur Massar", "Malika", "Yoff",
        "Rond-point EMG", "Bus Tata", "Cité imbécile"
    ],
    "Thiès": [
        "Mbour", "Saly", "Tivaouane", "Pout", "Joal", "Sindia", "Somone", 
        "Ndiass", "AIBD", "Khombole", "Meckhe"
    ],
    "Diourbel": [
        "Touba", "Mbacké", "Bambey"
    ],
    "Saint-Louis": [
        "Podor", "Richard-Toll", "Dagana", "Ndioum", "Ross Béthio", "Pété"
    ],
    "Ziguinchor": [
        "Bignona", "Oussouye", "Casamance", "Sindian", "Kafountine", "Blouf", "Mlomp", "Ediamath"
    ],
    "Kaolack": [
        "Nioro", "Passy", "Guinguinéo", "Kahone"
    ],
    "Fatick": [
        "Foundiougne", "Gossas", "Sokone", "Toubacouta"
    ],
    "Louga": [
        "Kébémer", "Linguère", "Dahra"
    ],
    "Kaffrine": [
        "Kaffrine", "Boulel", "Mbirkilane", "Sagna", "Koungheul", "Wengui"
    ],
    "Tambacounda": [
        "Goudiry", "Bakel", "Koupentoum", "DCSOM"
    ],
    "Matam": [
        "Ourossogui", "Kanel", "Ranérou", "Seddo Sebbé"
    ],
    "Sédhiou": [
        "Dianka"
    ]
}

In [ ]:
def deep_extract_city(row):
    # Si la ville est déjà trouvée, on la garde
    if row["ville"] != "Inconnue":
        return row["ville"]
    
    # On combine titre et description pour la recherche
    texte_complet = f"{str(row['titre'])} {str(row['description'])}".lower()
    
    # Parcours du dictionnaire contextuel
    for ville, mots_cles in mapping_contextuel.items():
        for mot in mots_cles:
            if mot.lower() in texte_complet:
                return ville
                
    return "Inconnue"

# Application de la fonction
df["ville"] = df.apply(deep_extract_city, axis=1)

In [ ]:
df.groupby("ville").size().sort_values(ascending=False)

In [ ]:
# Mots-clés qui indiquent un accident de la route
mots_routiers = [
    "collision", "percuté", "renversé", "choc", "dérapage", "tonneau", 
    "bus", "car", "moto", "véhicule", "camion", "scooter", "minibus", 
    "ndiaga ndiaye", "sept-place", "tata", "clando", "tipper", "poids lourd"
]

# Mots-clés à exclure (Bruit)
mots_exclusion = [
    "pirogue", "chavirement", "mer", "noyade", "mortier", "rebelle", 
    # "mfdc", "armée", "militaire", "élection", "manifestation"
]

In [ ]:
def filtrer_accidents_route(df):
    # 1. Mise en minuscule pour la comparaison
    df['texte_recherche'] = (df['titre'] + " " + df['description']).str.lower()
    
    # 2. On garde si un mot routier est présent
    masque_routier = df['texte_recherche'].apply(
        lambda x: any(mot in str(x) for mot in mots_routiers)
    )
    
    # 3. On exclut si un mot banni est présent (prioritaire)
    masque_exclusion = df['texte_recherche'].apply(
        lambda x: any(mot in str(x) for mot in mots_exclusion)
    )
    
    # Application du filtre : Routier ET NON Exclusion
    df_clean = df[masque_routier & ~masque_exclusion].copy()
    
    # Supprimer la colonne temporaire
    df_clean.drop(columns=['texte_recherche'], inplace=True)
    
    return df_clean

# Utilisation
df_accidents_route = filtrer_accidents_route(df)

print(f"Nombre d'articles avant : {len(df)}")
print(f"Nombre d'accidents routiers après filtrage : {len(df_accidents_route)}")

In [ ]:
df_accidents_route['date'] = pd.to_datetime(df_accidents_route['date'])

In [ ]:
df_accidents_route.groupby("ville").size().sort_values(ascending=False)

In [ ]:
# Filtrer pour n'avoir que les villes inconnues
df_inconnues = df_accidents_route[df_accidents_route["ville"] == "Inconnue"].copy()

# Afficher les premières lignes pour inspection (titre et description)
print(f"Il reste {len(df_inconnues)} lignes à identifier.")
pd.set_option('display.max_colwidth', 150) # Pour lire plus facilement les descriptions
print(df_inconnues[["titre", "description"]].head(20))

In [ ]:
# A supprimer après vérification manuelle
import re

def extract_gravite(text):
    if not isinstance(text, str):
        return 0, 0
    
    text = text.lower()
    # Dictionnaire pour convertir les petits nombres en lettres
    nombre_map = {
        "un": 1, "une": 1, "deux": 2, "trois": 3, "quatre": 4, 
        "cinq": 5, "six": 6, "sept": 7, "huit": 8, "neuf": 9, "dix": 10
    }
    
    def get_number(match_group):
        val = match_group.strip()
        if val.isdigit():
            return int(val)
        return nombre_map.get(val, 0)

    # Extraction des morts/décès
    morts = 0
    pattern_morts = r"(\d+|un|une|deux|trois|quatre|cinq|six|sept|huit|neuf|dix)\s*(mort|décès|vie)"
    match_morts = re.search(pattern_morts, text)
    if match_morts:
        morts = get_number(match_morts.group(1))
    
    # Extraction des blessés
    blesses = 0
    pattern_blesses = r"(\d+|un|une|deux|trois|quatre|cinq|six|sept|huit|neuf|dix)\s*(blessé|victime)"
    match_blesses = re.search(pattern_blesses, text)
    if match_blesses:
        blesses = get_number(match_blesses.group(1))
        
    return morts, blesses

# Application au DataFrame
# On combine titre et description pour maximiser les chances de trouver l'info
df_accidents_route['temp_gravite'] = (df_accidents_route['titre'] + " " + df_accidents_route['description']).apply(extract_gravite)

# Séparation en deux colonnes distinctes
df_accidents_route[['nb_morts', 'nb_blesses']] = pd.DataFrame(df_accidents_route['temp_gravite'].tolist(), index=df_accidents_route.index)
df_accidents_route.drop(columns=['temp_gravite'], inplace=True)

In [ ]:
import re

def extract_gravite_v3(row):
    # On nettoie le texte pour faciliter la recherche
    text = f"{str(row['titre'])} {str(row['description'])}".lower()
    
    # 1. Dictionnaire des quantités
    quantites = {
        "une vingtaine": 20, "une dizaine": 10, "une quinzaine": 15,
        "six": 6, "sept": 7, "quatre": 4, "trois": 3, "deux": 2, "un": 1, "une": 1
    }

    morts, blesses = 0, 0

    # 2. Extraction des Blessés (Priorité à la ligne 8)
    # On cherche d'abord les nombres (chiffres ou mots) suivis de "blessé"
    for mot, valeur in quantites.items():
        if re.search(rf"({valeur}|{mot})\s*(blessés?|graves?)", text):
            blesses = max(blesses, valeur)
    
    # On cherche aussi les chiffres purs (ex: "15 blessés")
    chiffres_blesses = re.findall(r"(\d+)\s*(blessés?)", text)
    for val, _ in chiffres_blesses:
        blesses = max(blesses, int(val))

    # 3. Extraction des Morts (Avec filtre anti-faux positifs)
    # On ne compte un mort "par défaut" que si c'est un récit d'accident réel
    mots_action = ["tue", "meurt", "perd la vie", "perdent la vie", "coûte la vie", "décédé", "fauché"]
    
    for mot, valeur in quantites.items():
        if re.search(rf"({valeur}|{mot})\s*(morts?|décès|vie|tués?)", text):
            morts = max(morts, valeur)
            
    if morts == 0 and any(exp in text for exp in mots_action):
        # Si on ne trouve pas de chiffre mais qu'un mot d'action est là : c'est 1 mort
        morts = 1

    return pd.Series([morts, blesses])

# Application
df_accidents_route[['nb_morts', 'nb_blesses']] = df_accidents_route.apply(extract_gravite_v3, axis=1)

In [ ]:
# 1. Supprimer les lignes où la ville est encore "Inconnue" (comme le Mali)
df_final = df[df["ville"] != "Inconnue"].copy()

# 2. Supprimer les doublons potentiels (si deux articles parlent du même accident)
df_final = df_final.drop_duplicates(subset=['titre'])

print(f"Ton dataset contient maintenant {len(df_final)} accidents qualifiés avec localisation et gravité.")

In [ ]:
# Vérifier les lignes où on a détecté des morts
print(df_accidents_route[df_accidents_route['nb_morts'] > 0][['titre', 'nb_morts', 'nb_blesses']].head())

In [ ]:
df_accidents_route.head(20)

In [ ]:
# Suppression des lignes "Bruit"
mots_bruit = ["espagne", "contrôle technique", "lutter contre", "prévention", "ministre rappelle"]
df_final = df_accidents_route[~df_accidents_route['titre'].str.lower().str.contains('|'.join(mots_bruit))]

# Suppression des morts à l'étranger (si ville identifiée est hors Sénégal)
# Tu peux ajouter "Espagne" ou "Mali" dans ton dictionnaire pour les filtrer ensuite.

In [ ]:
df_final

In [ ]:
# Ajouter ces donnees dans le CSV final a la fin du fichier existant accidents_senegal.csv
df_final.to_csv("accidents_senegal.csv", index=False, encoding="utf-8-sig", mode="a", header=False)

In [ ]:
expected_cols = [
    "date", "titre", "description", "lien",
    "ville", "nb_morts", "nb_blesses", "source"
]

# 1. S'assurer d’avoir toutes les colonnes
for c in expected_cols:
    if c not in df_final.columns:
        df_final[c] = ""  # ou 0 pour nb_morts/nb_blesses selon contexte

# 2. Réordonner selon le format cible
df_final = df_final.reindex(columns=expected_cols)

# 3. Écrire en append (header=False si déjà existant)
df_final.to_csv(
    "accidents_senegal.csv",
    index=False,
    encoding="utf-8-sig",
    mode="a",         # ou mode="w" si vous reconstruisez le fichier proprement
    header=False      # True si vous créez depuis zéro
)